# 02 Feature Engineering

This notebook loads the cleaned cryptocurrency datasets created in notebook `01a_data_collection_and_cleaning` and the macro dataset created in notebook `01b_macro_data_collection`. It processes the macro data and computes time-series features for later analysis and modeling.

The feature set is designed to capture both general market behavior and dependence on Bitcoin.

In particular, the notebook creates:

| Feature | Meaning                                                                                                            |
| --- |--------------------------------------------------------------------------------------------------------------------|
| **daily log returns** | How much did the price change from yesterday to today?                                                             |
| **rolling volatility** | How much does this coin fluctuate?                                                                                 |
| **rolling correlation with Bitcoin** | Does this coin move together with Bitcoin?                                                                         |
| **idiosyncratic volatility** based on residuals from a regression on Bitcoin returns | How much of this coin’s movement is not explained by Bitcoin?                                                      |
| **volume-based features** | How reliable is the coin? <br> `high volume`: strong market activity <br> `low volume`: illiquid and less reliable |
| **macro log returns** (e.g. VIX, DXY) | How do broader market conditions change over time, independent of crypto prices? |


In [24]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration

This section defines the input and output paths used in the notebook.

In [25]:
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Full historical wide datasets
FULL_CLOSE_INPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_close_full.csv"
FULL_VOLUME_INPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_volume_full.csv"

# Aligned modeling datasets
ALIGNED_CLOSE_INPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_close_aligned.csv"
ALIGNED_VOLUME_INPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_volume_aligned.csv"

# Aligned macro dataset
MACRO_ALIGNED_INPUT_PATH = DATA_PROCESSED_DIR / "macro_wide_close_aligned.csv"
MACRO_ALIGNED_FILLED_OUTPUT_PATH = DATA_PROCESSED_DIR / "macro_wide_close_aligned_filled.csv"

# Feature datasets in long format (one row per asset and date)
FEATURES_LONG_FULL_OUTPUT_PATH = DATA_PROCESSED_DIR / "crypto_features_long_full.csv"
FEATURES_LONG_ALIGNED_OUTPUT_PATH = DATA_PROCESSED_DIR / "crypto_features_long_aligned.csv"

# Wide-format log returns used for modeling
FEATURES_WIDE_RETURNS_OUTPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_log_returns_aligned.csv"
MACRO_WIDE_RETURNS_OUTPUT_PATH = DATA_PROCESSED_DIR / "macro_wide_log_returns_aligned.csv"

# Summary of missing values and data quality in the feature datasets
FEATURES_SUMMARY_OUTPUT_PATH = DATA_PROCESSED_DIR / "feature_quality_summary.csv"

VOLATILITY_WINDOW = 30
CORRELATION_WINDOW = 30
LONG_CORRELATION_WINDOW = 90
IDIO_WINDOW = 30

## Load and preprocess macro datasets

The aligned macro dataset is loaded and prepared for feature engineering.

Because macro factors follow market trading calendars, missing values occur on weekends and holidays when no observations are available. To ensure compatibility with the crypto dataset, missing values are forward-filled.

This assumes that macro conditions remain constant between trading days and allows the macro data to be aligned with the continuous crypto time series.

In [26]:
macro_aligned_df = pd.read_csv(MACRO_ALIGNED_INPUT_PATH, parse_dates=["Date"], index_col="Date")

macro_aligned_df = macro_aligned_df.ffill()

In [27]:
print("Missing values after forward fill:")
print(macro_aligned_df.isna().sum())

Missing values after forward fill:
DXY    3
VIX    3
dtype: int64


<div class="alert alert-block alert-info">
After forward filling, a small number of missing values may remain at the beginning of the dataset if no prior observations are available. These initial rows will be removed in later preprocessing steps.
</div>

### Compute macro log returns

To integrate macroeconomic market factors into the feature set, daily log returns are computed for the aligned macro series. These returns can later be used as additional explanatory variables alongside the crypto-specific features.

In [28]:
macro_log_returns_df = np.log(macro_aligned_df / macro_aligned_df.shift(1))
macro_log_returns_df.head()

,DXY,VIX
Date,,
2020-04-10,NaN,NaN
2020-04-11,NaN,NaN
2020-04-12,NaN,NaN
2020-04-13,NaN,NaN
2020-04-14,-0.00455,-0.086459


## Load cleaned crypto datasets

The full datasets preserve the full historical range, while the aligned datasets only contain dates for which all selected assets are available.

In [29]:
close_full_df = pd.read_csv(FULL_CLOSE_INPUT_PATH, parse_dates=["Date"], index_col="Date")
volume_full_df = pd.read_csv(FULL_VOLUME_INPUT_PATH, parse_dates=["Date"], index_col="Date")

close_aligned_df = pd.read_csv(ALIGNED_CLOSE_INPUT_PATH, parse_dates=["Date"], index_col="Date")
volume_aligned_df = pd.read_csv(ALIGNED_VOLUME_INPUT_PATH, parse_dates=["Date"], index_col="Date")

In [30]:
print("Full close shape:", close_full_df.shape)
print("Full volume shape:", volume_full_df.shape)
print("Aligned close shape:", close_aligned_df.shape)
print("Aligned volume shape:", volume_aligned_df.shape)

Full close shape: (3362, 6)
Full volume shape: (3362, 6)
Aligned close shape: (2167, 6)
Aligned volume shape: (2167, 6)


## Compute log returns

Log returns are used instead of raw prices because they are more suitable for time-series analysis and make price changes comparable across assets with different price levels.

In [31]:
def compute_log_returns(price_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute daily log returns from a wide-format price DataFrame.
    """
    return np.log(price_df / price_df.shift(1))

In [32]:
log_returns_full_df = compute_log_returns(close_full_df)
log_returns_aligned_df = compute_log_returns(close_aligned_df)

log_returns_aligned_df.head()

,BNB,BTC,ETH,SOL,TRX,XRP
Date,,,,,,
2020-04-10,NaN,NaN,NaN,NaN,NaN,NaN
2020-04-11,0.004834,-0.000934,-0.001241,-0.202363,0.001684,0.003190
2020-04-12,0.040519,0.016198,0.018327,0.127559,0.033015,0.010957
2020-04-13,0.044844,-0.018248,-0.030642,-0.126256,-0.024877,-0.012719
2020-04-14,0.032114,-0.000381,0.008391,-0.161358,-0.013761,-0.012850


<div class="alert alert-block alert-info">
The first row of the dataset contains NaN values since it's the first date and there are no previous values to calculate from. These rows will be cleaned out later on.
</div>

## Compute rolling volatility

Rolling volatility is estimated as the standard deviation of log returns over a fixed time window.

In [33]:
def compute_rolling_volatility(return_df: pd.DataFrame, window: int) -> pd.DataFrame:
    """
    Compute rolling volatility from log returns.
    """
    return return_df.rolling(window=window).std()

In [34]:
rolling_volatility_full_df = compute_rolling_volatility(log_returns_full_df, VOLATILITY_WINDOW)
rolling_volatility_aligned_df = compute_rolling_volatility(log_returns_aligned_df, VOLATILITY_WINDOW)

## Compute rolling correlation with Bitcoin

Rolling correlation with Bitcoin is one of the main dependence features in this project. It measures how strongly each asset moves together with Bitcoin over time.

In [35]:
def compute_rolling_correlation_with_btc(return_df: pd.DataFrame, window: int, btc_col: str = "BTC") -> pd.DataFrame:
    """
    Compute rolling correlation of each asset with Bitcoin.
    """
    btc_series = return_df[btc_col]
    corr_df = pd.DataFrame(index=return_df.index)

    for col in return_df.columns:
        corr_df[col] = return_df[col].rolling(window=window).corr(btc_series)

    return corr_df

In [36]:
rolling_corr_30_full_df = compute_rolling_correlation_with_btc(log_returns_full_df, CORRELATION_WINDOW)
rolling_corr_30_aligned_df = compute_rolling_correlation_with_btc(log_returns_aligned_df, CORRELATION_WINDOW)

rolling_corr_90_full_df = compute_rolling_correlation_with_btc(log_returns_full_df, LONG_CORRELATION_WINDOW)
rolling_corr_90_aligned_df = compute_rolling_correlation_with_btc(log_returns_aligned_df, LONG_CORRELATION_WINDOW)

## Compute idiosyncratic volatility

Idiosyncratic volatility is used here as an estimate of asset-specific risk that is not explained by Bitcoin.


In [37]:
def compute_idiosyncratic_volatility(
    return_df: pd.DataFrame,
    window: int,
    btc_col: str = "BTC",
    ) -> pd.DataFrame:
    """
    Compute rolling idiosyncratic volatility for each asset relative to Bitcoin.

    For each rolling window, the asset return is regressed on Bitcoin return using
    ordinary least squares. The standard deviation of the residuals is then used
    as the idiosyncratic volatility.
    """
    btc_returns = return_df[btc_col]
    output_df = pd.DataFrame(index=return_df.index, columns=return_df.columns, dtype=float)

    for asset in return_df.columns:
        if asset == btc_col:
            output_df[asset] = 0.0
            continue

        asset_series = return_df[asset]
        values = []

        for i in range(len(return_df)):
            if i < window - 1:
                values.append(np.nan)
                continue

            y = asset_series.iloc[i - window + 1 : i + 1]
            x = btc_returns.iloc[i - window + 1 : i + 1]

            window_df = pd.DataFrame({"y": y, "x": x}).dropna()

            if len(window_df) < window:
                values.append(np.nan)
                continue

            x_vals = window_df["x"].to_numpy()
            y_vals = window_df["y"].to_numpy()

            x_mean = x_vals.mean()
            y_mean = y_vals.mean()

            denom = ((x_vals - x_mean) ** 2).sum()
            if denom == 0:
                values.append(np.nan)
                continue

            beta = ((x_vals - x_mean) * (y_vals - y_mean)).sum() / denom
            alpha = y_mean - beta * x_mean

            residuals = y_vals - (alpha + beta * x_vals)
            values.append(np.std(residuals, ddof=1))

        output_df[asset] = values

    return output_df

In [38]:
idiosyncratic_vol_full_df = compute_idiosyncratic_volatility(log_returns_full_df, IDIO_WINDOW)
idiosyncratic_vol_aligned_df = compute_idiosyncratic_volatility(log_returns_aligned_df, IDIO_WINDOW)

### Mathematical explanation

For each asset and each rolling window of length $W$, a simple linear regression is estimated:

$$
r_{i,t} = \alpha_i + \beta_i \,  \times r_{BTC,t} + \varepsilon_{i,t}
$$

where:

- $r_{i,t}$: log return of asset $i$ at time $t$
- $\alpha_i$: baseline return independent of Bitcoin
- $\beta_i$: sensitivity of the asset to Bitcoin
- $r_{BTC,t}$: log return of Bitcoin at time $t$
- $\varepsilon_{i,t}$: residual, the unexplained component

#### Estimation within a rolling window

For each window, the regression parameters are estimated using ordinary least squares:

$$
\beta_i =
\frac{\sum_{t=1}^{W}(x_t - \bar{x})(y_t - \bar{y})}
{\sum_{t=1}^{W}(x_t - \bar{x})^2}
$$

$$
\alpha_i = \bar{y} - \beta_i \bar{x}
$$

where:

- $x_t = r_{BTC,t}$
- $y_t = r_{i,t}$
- $\bar{x}$ and $\bar{y}$ are the mean values within the rolling window

#### Residuals

The residuals represent the part of the asset return that is not explained by Bitcoin:

$$
\varepsilon_{i,t} = y_t - (\alpha_i + \beta_i x_t)
$$

#### Idiosyncratic volatility

The idiosyncratic volatility is defined as the standard deviation of the residuals within the rolling window:

$$
\sigma^{idio}_i =
\sqrt{\frac{1}{W-1}\sum_{t=1}^{W}(\varepsilon_{i,t} - \bar{\varepsilon})^2}
$$

#### Interpretation

| Idiosyncratic Volatility | Meaning |
| --- | --- |
| High | the asset shows stronger asset-specific behavior and is less explained by Bitcoin |
| Low | the asset is more strongly driven by Bitcoin movements |

In this project, idiosyncratic volatility is used as an indicator of how independent an altcoin behaves relative to Bitcoin.

## Volume-based features

In addition to price-based features, simple volume features are included in the form of log-transformed volume and daily log volume change. Including volume-based features allows the model to account for variations in market activity that are not reflected in price changes alone. \
For example, a price increase accompanied by high trading volume may indicate strong market participation, whereas the same price increase with low volume may be less reliable and driven by fewer market participants.

In [39]:
def compute_log_volume(volume_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute log-transformed trading volume.
    """
    return np.log1p(volume_df)


def compute_log_volume_change(volume_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute daily change in log-transformed trading volume.
    """
    log_volume_df = np.log1p(volume_df)
    return log_volume_df.diff()

In [40]:
log_volume_full_df = compute_log_volume(volume_full_df)
log_volume_aligned_df = compute_log_volume(volume_aligned_df)

log_volume_change_full_df = compute_log_volume_change(volume_full_df)
log_volume_change_aligned_df = compute_log_volume_change(volume_aligned_df)

## Crypto & Macro dataset safety check
Confirm that the macro and crypto indexes match before aligning them.

In [41]:
print("Crypto aligned index equals macro aligned index:", close_aligned_df.index.equals(macro_log_returns_df.index))

Crypto aligned index equals macro aligned index: True


## Convert wide features to long format

A long-format feature table is created so that each row corresponds to one asset on one date. This format is convenient for clustering, visualization, and later export.

In [42]:
def wide_to_long_feature(wide_df: pd.DataFrame, feature_name: str) -> pd.DataFrame:
    """
    Convert a wide-format feature DataFrame to long format.
    """
    long_df = (
        wide_df
        .reset_index()
        .melt(id_vars="Date", var_name="Ticker", value_name=feature_name)
        .sort_values(["Date", "Ticker"])
        .reset_index(drop=True)
    )
    return long_df

In [43]:
def build_feature_table(
    close_df: pd.DataFrame,
    return_df: pd.DataFrame,
    volatility_df: pd.DataFrame,
    corr_30_df: pd.DataFrame,
    corr_90_df: pd.DataFrame,
    idio_df: pd.DataFrame,
    log_volume_df: pd.DataFrame,
    log_volume_change_df: pd.DataFrame,
    macro_return_df: pd.DataFrame | None = None,
    ) -> pd.DataFrame:
    """
    Merge all feature DataFrames into one long-format feature table.
    """
    feature_tables = [
        wide_to_long_feature(close_df, "close"),
        wide_to_long_feature(return_df, "log_return"),
        wide_to_long_feature(volatility_df, f"volatility_{VOLATILITY_WINDOW}d"),
        wide_to_long_feature(corr_30_df, f"btc_corr_{CORRELATION_WINDOW}d"),
        wide_to_long_feature(corr_90_df, f"btc_corr_{LONG_CORRELATION_WINDOW}d"),
        wide_to_long_feature(idio_df, f"idio_vol_{IDIO_WINDOW}d"),
        wide_to_long_feature(log_volume_df, "log_volume"),
        wide_to_long_feature(log_volume_change_df, "log_volume_change"),
    ]

    merged_df = feature_tables[0]
    for df in feature_tables[1:]:
        merged_df = merged_df.merge(df, on=["Date", "Ticker"], how="left")

    if macro_return_df is not None:
        macro_feature_df = macro_return_df.reset_index().copy()
        macro_feature_df.columns = ["Date"] + [f"{col.lower()}_log_return" for col in macro_return_df.columns]
        merged_df = merged_df.merge(macro_feature_df, on="Date", how="left")

    return merged_df

In [44]:
features_long_full_df = build_feature_table(
    close_df=close_full_df,
    return_df=log_returns_full_df,
    volatility_df=rolling_volatility_full_df,
    corr_30_df=rolling_corr_30_full_df,
    corr_90_df=rolling_corr_90_full_df,
    idio_df=idiosyncratic_vol_full_df,
    log_volume_df=log_volume_full_df,
    log_volume_change_df=log_volume_change_full_df,
    macro_return_df=None,
)

features_long_aligned_df = build_feature_table(
    close_df=close_aligned_df,
    return_df=log_returns_aligned_df,
    volatility_df=rolling_volatility_aligned_df,
    corr_30_df=rolling_corr_30_aligned_df,
    corr_90_df=rolling_corr_90_aligned_df,
    idio_df=idiosyncratic_vol_aligned_df,
    log_volume_df=log_volume_aligned_df,
    log_volume_change_df=log_volume_change_aligned_df,
    macro_return_df=macro_log_returns_df,
)

features_long_aligned_df.head()

,Date,Ticker,close,log_return,volatility_30d,btc_corr_30d,btc_corr_90d,idio_vol_30d,log_volume,log_volume_change,dxy_log_return,vix_log_return
0,2020-04-10,BNB,13.737724,NaN,NaN,NaN,NaN,NaN,19.792704,NaN,NaN,NaN
1,2020-04-10,BTC,6865.493164,NaN,NaN,NaN,NaN,0.0,24.498847,NaN,NaN,NaN
2,2020-04-10,ETH,158.412445,NaN,NaN,NaN,NaN,NaN,23.612578,NaN,NaN,NaN
3,2020-04-10,SOL,0.951054,NaN,NaN,NaN,NaN,NaN,18.285597,NaN,NaN,NaN
4,2020-04-10,TRX,0.012462,NaN,NaN,NaN,NaN,NaN,20.816288,NaN,NaN,NaN


## Inspect missing values in the feature set

Some missing values are expected at the beginning of the time series because rolling statistics require a minimum number of observations.

In [45]:
feature_quality_summary = pd.DataFrame({
    "missing_full": features_long_full_df.isna().sum(),
    "missing_aligned": features_long_aligned_df.isna().sum(),
})

feature_quality_summary

,missing_full,missing_aligned
Date,0.0,0
Ticker,0.0,0
btc_corr_30d,2623.0,180
btc_corr_90d,2983.0,540
close,2443.0,0
dxy_log_return,NaN,24
idio_vol_30d,2593.0,150
log_return,2449.0,6
log_volume,2443.0,0
log_volume_change,2449.0,6


In [46]:
feature_quality_summary.to_csv(FEATURES_SUMMARY_OUTPUT_PATH, index=True)
print(f"Saved feature quality summary to: {FEATURES_SUMMARY_OUTPUT_PATH}")

Saved feature quality summary to: /home/theodora/PycharmProjects/HSLU_FS25_DSPRO2/data/processed/feature_quality_summary.csv


## Modeling subset

For some downstream methods, it may be useful to remove rows with incomplete features after the rolling windows have been applied.

In [47]:
features_long_aligned_complete_df = features_long_aligned_df.dropna().copy()

print("Aligned feature table shape before dropping NaNs:", features_long_aligned_df.shape)
print("Aligned feature table shape after dropping NaNs:", features_long_aligned_complete_df.shape)

Aligned feature table shape before dropping NaNs: (13002, 12)
Aligned feature table shape after dropping NaNs: (12462, 12)


## Save outputs

The main outputs of this notebook are:

- forward-filled aligned macro dataset
- full long-format feature dataset
- aligned long-format feature dataset
- aligned wide-format log returns
- feature quality summary

In [48]:
macro_aligned_df.to_csv(MACRO_ALIGNED_FILLED_OUTPUT_PATH, index=True)
macro_log_returns_df.to_csv(MACRO_WIDE_RETURNS_OUTPUT_PATH, index=True)
features_long_full_df.to_csv(FEATURES_LONG_FULL_OUTPUT_PATH, index=False)
features_long_aligned_df.to_csv(FEATURES_LONG_ALIGNED_OUTPUT_PATH, index=False)
log_returns_aligned_df.to_csv(FEATURES_WIDE_RETURNS_OUTPUT_PATH, index=True)

print(f"Saved forward-filled macro dataset to: {MACRO_ALIGNED_FILLED_OUTPUT_PATH}")
print(f"Saved aligned macro log returns to: {MACRO_WIDE_RETURNS_OUTPUT_PATH}")
print(f"Saved full long-format features to: {FEATURES_LONG_FULL_OUTPUT_PATH}")
print(f"Saved aligned long-format features to: {FEATURES_LONG_ALIGNED_OUTPUT_PATH}")
print(f"Saved aligned wide-format log returns to: {FEATURES_WIDE_RETURNS_OUTPUT_PATH}")

Saved forward-filled macro dataset to: /home/theodora/PycharmProjects/HSLU_FS25_DSPRO2/data/processed/macro_wide_close_aligned_filled.csv
Saved aligned macro log returns to: /home/theodora/PycharmProjects/HSLU_FS25_DSPRO2/data/processed/macro_wide_log_returns_aligned.csv
Saved full long-format features to: /home/theodora/PycharmProjects/HSLU_FS25_DSPRO2/data/processed/crypto_features_long_full.csv
Saved aligned long-format features to: /home/theodora/PycharmProjects/HSLU_FS25_DSPRO2/data/processed/crypto_features_long_aligned.csv
Saved aligned wide-format log returns to: /home/theodora/PycharmProjects/HSLU_FS25_DSPRO2/data/processed/crypto_wide_log_returns_aligned.csv
